# PROJECT 1 · STEP 4 — Physics-informed DOE Modeling

> 결과는 synthetic Engineering Scenario이며 실제 Fab recipe나 생산 개선 실적이 아니다.

In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
doe = pd.read_csv(ROOT / 'data/processed/doe_results.csv')
effects = pd.read_csv(ROOT / 'results/doe_effects.csv')
ranked = pd.read_csv(ROOT / 'results/doe_ranked_conditions.csv')
summary = json.loads((ROOT / 'results/doe_summary.json').read_text(encoding='utf-8'))
summary

## 1. Physics features

`process_margin_s`, `shift_safety_factor`, `contact_angle_deg`, `interface_shear_mpa`는 원인 해석을 위한 mechanism response다. 계수는 project scenario이며 실제 material characterization으로 재보정해야 한다.

In [ ]:
doe[['process_margin_s','shift_safety_factor','contact_angle_deg','interface_shear_mpa','edge_void_pct','chip_offset_p95_um']].corr().round(2)

## 2. Factor and interaction effects

Whole-plot factor EMC/roughness는 whole plot 4개뿐이므로 계수 방향만 탐색적으로 해석한다.

In [ ]:
effects.query("term != 'intercept'").sort_values(['response','high_minus_low_effect'], key=lambda s: s.abs(), ascending=False).head(20)

## 3. Multi-response decision

Void와 offset에는 각각 2배 weight, warpage와 cycle time에는 1배 weight를 준 geometric desirability를 사용한다. Weight 변경 민감도는 실제 제품 우선순위에 맞춰 재검토한다.

In [ ]:
ranked[['run_order','emc_lot','film_roughness_class','vacuum_base_kpa_abs','zone_range_c','closing_speed_mm_s','edge_void_pct','chip_offset_p95_um','warpage_um','cycle_time_index','overall_desirability']].head(10)

## Decision

상위 조건은 실제 양산 recipe가 아니라 confirmation 후보이다. 다른 chamber/material lot, measurement repeatability와 short-shot physical signature를 통과해야 root cause와 개선안으로 승격한다.